In [ ]:
# Env
NUM_AGENTS = 4
HEIGHT = 9
WIDTH = 9
SPAWN_PROB_PER_CELL = 0.05
DESPAWN_PROB_PER_CELL = 0.09

# Training
DISCOUNT = 0.99
EPSILON = 0.1
LEARNING_RATE = 0.00002
BATCH_SIZE = 32
TARGET_UPDATE_FREQUENCY = 100

# Model
CONV_CHANNELS = [16, 32]
HIDDEN_FEATURES = 128
HIDDEN_LAYERS = 2
KERNEL_SIZE = 3


In [ ]:
import sys
sys.path.append('../../../')
from models.value_cnn import ValueCNNDecentralized, ValueCNNCentralized
import torch
from tadd_helpers.env_functions import State
import numpy as np
import pickle
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd

In [ ]:
decentralized_model_path = f"decentralized_model{WIDTH}x{HEIGHT}_agents{NUM_AGENTS}/decentralized_value_cnn_agent_i.pt"
decentralized_model_paths = []
for i in range(NUM_AGENTS):
    decentralized_model_paths.append(decentralized_model_path.replace(f"agent_i", f"agent_{i}"))
print(decentralized_model_paths)

centralized_model_2a_path = f"centralized_model{WIDTH}x{HEIGHT}_agents{NUM_AGENTS}/centralized_value_cnn.pt"

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
decentralized_2a_cnns: list[ValueCNNDecentralized] = []
for i in range(len(decentralized_model_paths)):
    model_path = decentralized_model_paths[i]
    cnn = ValueCNNDecentralized(HEIGHT, WIDTH, LEARNING_RATE, DISCOUNT, HIDDEN_FEATURES, HIDDEN_LAYERS, CONV_CHANNELS, KERNEL_SIZE)
    
    state_dict = torch.load(model_path, map_location=device)
    cnn.load_state_dict(state_dict)
    # --------------------
    
    # 3. Explicitly move the entire model to the chosen device
    cnn.to(device)

    cnn.eval()
    decentralized_2a_cnns.append(cnn)
centralized_2a_cnn = ValueCNNCentralized(HEIGHT, WIDTH, LEARNING_RATE, DISCOUNT, HIDDEN_FEATURES, HIDDEN_LAYERS, CONV_CHANNELS, KERNEL_SIZE)
centralized_2a_cnn.load_state_dict(torch.load(centralized_model_2a_path, map_location=device))
centralized_2a_cnn.to(device)
centralized_2a_cnn.eval()


In [ ]:
states: list[State] = []
states_file_path = f"centralized_model{WIDTH}x{HEIGHT}_agents{NUM_AGENTS}/trained_states_centralized.pkl"
with open(states_file_path, "rb") as f:
    states = pickle.load(f)
print(states[:3])

In [ ]:
print(len(states))
states_to_eval = states

In [ ]:
all_results = []
for t, state in tqdm(enumerate(states_to_eval), total=len(states_to_eval), desc="Evaluating states"):
    val_d = []
    for i in range(NUM_AGENTS):
        cnn = decentralized_2a_cnns[i]
        raw_dict = cnn.state_to_raw_dict(state)
        pred_value = cnn.get_model_reward_prediction_from_raw(raw_dict, agent_pos=state.agent_position(i))
        val_d.append(pred_value.item())
        
    valC = centralized_2a_cnn.get_model_reward_prediction_from_raw(centralized_2a_cnn.state_to_raw_dict(state)).item()
    
    val_d_team = sum(val_d)
    result_info = {
        "state_index": t,
        "timestep": (t / 2),
        "centralized_team_value": valC,
        "decentralized_team_value": val_d_team,
        "error": abs(valC - val_d_team),
        "centralized - decentralized": valC - val_d_team,
        "state_object": state
        
    }
    all_results.append(result_info)

In [ ]:
df = pd.DataFrame(all_results)
print(df.head())

In [ ]:
with open(f"predicted_trajectory_values_{WIDTH}x{HEIGHT}_agents{NUM_AGENTS}.pkl", "wb") as f:
    pickle.dump(df, f)